# 0 Introduction
- Import libraries and load the dataset into a dataframe for analysis.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("student-mat.csv", sep=";")

---

# 1 Exploratory Data Analysis

## Summary Statistics
- We calculated summary statistics to quickly understand the grade distribution before creating visual representations. The results are calculated below.

In [ ]:
finalGrades=df["G3"].values

# Calculate summary stats
mean = np.mean(finalGrades)
median = np.median(finalGrades)
standardDeviation = np.std(finalGrades)

print(f"Summary stastics:\n\tMean: {mean}\n\tMedian: {median}\n\tStandard Deviation: {standardDeviation}")

## Visualizations

### Distribution of final gades - Histogram
- We constructed a histogram to visualize the distribution of final grades for all students

In [ ]:
sns.set_theme()
histogram = sns.displot(data=df, x="G3")

- The grade distribution appears to be bimodal, with a bell shaped cluster centered around the median and a cohort of students scoring zero points.
---

#### Distribution of failures

In [ ]:
sns.barplot(data=df, x = "failures", y = "G3")

- Most students have zero failures, there appears to be a negative correlation between the number of failures and the number of students receving that number of failures. Or the number of students decreases as the number of failures increases.

### Comparison of Schools



In [ ]:
gpFinal = df[df["school"] == "GP"]["G3"]
msFinal = df[df["school"] == "MS"]["G3"]


print(f"Gabriel Pereira Statistics:\n\t- median final grade: {np.median(gpFinal)}\n\t- mean final grade: {np.mean(gpFinal)}\n\t- standard deviation of final grades: {np.std(gpFinal)}\n\t- sample size: {gpFinal.count()}")
print(f"Mousinho da Silveira:\n\t- median final grade: {np.median(msFinal)}\n\t- mean final grade: {np.mean(msFinal)}\n\t- standard deviation of final grades: {np.std(msFinal)}\n\t- sample size: {msFinal.count()}")
sns.barplot(df, x="school", y="G3")
plt.show()

### Relationship between semester and final grades

In [ ]:
# Rearrange dataframe for bivariate scatterplot
melted_df = df.melt(
    id_vars=["G3"],
    value_vars=["G1", "G2"],
    var_name="Grade_Type",
    value_name="Grade_Value",
)

# Single scatterplot call using hue
sns.scatterplot(
    data=melted_df, x="Grade_Value", y="G3", hue="Grade_Type", alpha=0.7
)

plt.xlabel("Semesterly Grade")
plt.ylabel("Final Grade")
plt.title("Semesters One & Two vs Final Grade")
plt.show()

There appears to be a positive correlation between semesterly grades with final grades.

#### Mean final grade and intent on higher education

In [ ]:
sns.barplot(data=df, x = "higher", y = "G3")
plt.title("Mean Final Grade by Intent on Higher Education")
plt.xlabel("Intend to pursue highed education")
plt.ylabel("Final grade")

- Students who intend to pursue higher education consistently out perform the students who do not. Of those intending to pursue higher education also have a much lower variability in final grade.

- However 375 of the students intend to pursue higher education, leaving 20 who do not. This is not a statistically representative sample and should be discarded for training.

#### Mean final grade for each category by attribute
- This loop iterates through all the attributes then groups the data into subsets to finds the mean for G3 for all of the possible values.
- From a quick glance, we can see that most attibute values correlate very little with G3 scores except for Absences and Student Intent to pursue Higher Education.

In [ ]:
for attribute in df.columns:
    if attribute == "G3" or attribute == "G1" or attribute == "G2":
        continue
    print(df.groupby(attribute)["G3"].mean().to_string() + "\n")

---

# 2 Data Preprocessing
- We normalized the education of the parents to account for different family structures. For two parent households we average the parents education and in single parent households the value of the custodial parent is assigned.
- True/False categories were encoded as strings, either "yes" or "no", which are non-numeric. We implemented a function to translate these to a binary 1 or 0 which can be correlated with grades.
	- Categories include: extracurricular activities, paid supplemental classes, higher education goals, school support, family support, and nursery attendance.
- The normalization functions are displayed below:

In [ ]:
# categories in data set which have true/false values
binaryCategories = ["activities", "paid", "higher", "schoolsup", "famsup", "nursery", "internet", "romantic"]

def normalizeParentalEducation(row):
	if row["Pstatus"] == "T":
		return (row["Fedu"] + row["Medu"])/2
	elif row["guardian"] == "mother":
		return row["Medu"]
	elif row["guardian"] == "father":
		return row["Fedu"]
	else:
		return 0

def tfToBinary(row, cat):
		if cat in binaryCategories:
			if type(row[cat]) == type("string"):
				if row[cat].strip() == "no":
					return 0
				elif row[cat].strip() == "yes":
						return 1
			else:
				raise ValueError(f"Expected type {type("cat")} and instead received {row[cat]} of type {type(row[cat])} while processing class {cat}")
		else:
			raise ValueError(f"Category {cat} not in list: {binaryCategories}")
		
# Normalize parents education
df["normParentEdu"] = df.apply(normalizeParentalEducation, axis=1)
# discard input categories after normalization
df = df.drop(columns=["Medu","Fedu"]) 

# applie true-false -> binary conversion for all relevant categories
for cat in binaryCategories:
	df[cat] = df.apply(tfToBinary, cat=cat, axis=1)


- After normalizing the data set, we performed a Pearson's correlation analysis of each numerical category against the the final grade. We used this to select the categories to train on in subsequent steps.

In [ ]:
correlations = df.corr(method="pearson", numeric_only=True)["G3"]
print(correlations.sort_values(ascending=False))

#print(f"total: {df["higher"].sum()} of {df["higher"].count()}") # compute numbers of higher vs non higher

- Our correlation analysis found that the preceeding semester averages had the strongest correlation with the final grade at 90% and 80% respectively. This confirms the trend displayed by the prior graph.
- The number of past failures displayed a negative correlation of 36%. Our normalized parental education had an 18.82% correlation, the remaining attributes had diminishing correlation with final grades.
---

# 3 Model Training
- We used the sklearn machine learning Python library because of the pre-built tools it offers
- We used a linear regression model based upon semester one and two grades.
  - Reasons:
    - G2 and G3 showed the highest correlations to G3.
    - No need to complicate the model and potentially overfit the data. 
- We split our data set 80-20% for the training and testing sets.

In [ ]:
inputTrainSet, inputTestSet, finalGradeTrainSet, finalGradeTestSet = train_test_split(df[["G1", "G2"]], df["G3"], test_size=0.2, random_state=42)
model = LinearRegression().fit(inputTrainSet, finalGradeTrainSet)
finalGradePredictions = model.predict(inputTestSet)

print(type(finalGradePredictions))

print("R^2:", r2_score(finalGradeTestSet, finalGradePredictions))
print("MSE:", mean_squared_error(finalGradeTestSet, finalGradePredictions))

---

# 4 Model Performance Visualization
- Ipsum Lorem

In [ ]:
resultDf = pd.DataFrame({"finalGradePredictions" : finalGradePredictions, 'finalGradeTestSet' : finalGradeTestSet, "absDifference" : np.abs(finalGradePredictions - finalGradeTestSet)})

sns.scatterplot(resultDf, x="finalGradePredictions", y="finalGradeTestSet")
plt.errorbar(
    x=resultDf["finalGradePredictions"],
    y=resultDf["finalGradeTestSet"],
    yerr=resultDf["absDifference"],  # Use yerr=resultDf["absDifference"] for vertical error bars
    fmt="none",  # Don't plot new markers on top of seaborn scatter
    ecolor="lightgray",  # Light color so it doesn't obscure points
    alpha=0.6,
    capsize=3,  # Adds small horizontal caps to the end of error bars
)
plt.xlabel("Predicted Final Grade")
plt.ylabel("Actual Final Grade")
plt.title("Predictions vs Actuals with Absolute Error")
plt.show()